# 08 — Evaluation Quantitativa di DiscoverAI

**Obiettivo**: rispondere alle domande che `project_rulez.pdf` pone esplicitamente
(*quali metriche state usando? quali baseline? avete fatto hyperparameter tuning?*)
con numeri reali su un evaluation set definito *a priori*.

> **Caveat metodologico chiave**: questo è un *pseudo-relevance benchmark* costruito
> tramite keyword matching su `product_text_base` (title + brand + features +
> description). Tende a favorire i sistemi lessicali (BM25). I numeri assoluti
> sono **stime relative**, non ground truth. Per un benchmark "vero" servirebbero
> annotazioni umane su 100+ query (vedi conclusioni).

## Cosa è dentro

1. Setup e qrels (38 query, pseudo-relevance a 4 livelli)
2. **Confronto principale** dei 5 sistemi
3. **Ablation 1**: peso del re-ranking (β_quality, β_popularity)
4. **Ablation 2**: solo popularity score → trovato il responsabile della regressione MRR
5. **Ablation 3**: moduli di `search_v3` (negation, min_rating)
6. **Per-intent**: dove ogni sistema vince
7. **MRR top-1 analysis**: perché il re-ranking peggiora il primo risultato
8. **Summarization & entity coverage**
9. **Guardrail confusion matrix**
10. **Patch suggerite** + raccomandazioni finali


## 1. Setup e qrels


In [ ]:
import os, sys, re, pickle, json
import numpy as np, pandas as pd, faiss

ROOT = "/path/to/Mean-Squared-Terrors"
sys.path.insert(0, os.path.join(ROOT, "src"))

from mean_squared_terrors import search as team_search
from mean_squared_terrors import search_extended as team_extended
from mean_squared_terrors.eval_set import EVAL_QUERIES
from mean_squared_terrors.eval_metrics import (build_qrels, ndcg_at_k, mrr, recall_at_k,
                                               precision_at_k, average_precision, aggregate_metrics)
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

faiss_idx = faiss.read_index(f"{ROOT}/data/faiss_index.bin")
idx_df    = pd.read_csv(f"{ROOT}/data/embedding_index_enriched.csv")
products  = pd.read_csv(f"{ROOT}/data/products_cleaned.csv")
catalog   = idx_df.merge(products[["parent_asin", "product_text_base"]],
                          on="parent_asin", how="left")
qrels = build_qrels(catalog, EVAL_QUERIES)
print(f"Catalog: {len(catalog):,} products | Queries: {len(EVAL_QUERIES)}")


## 2. Confronto principale dei 5 sistemi


In [ ]:
summary = pd.read_csv(f"{ROOT}/data/eval_output/summary.csv")
print(summary.round(4).to_string(index=False))


### Lettura

- **BM25** è competitivo o superiore su NDCG, Precision, MAP. Parte è bias del benchmark
  (keyword matching favorisce BM25), parte è verità (BM25 batte anche embedding model
  su query con keyword esatte).
- **`semantic` puro** ha l'**MRR più alto** (0.855): il primo risultato è quasi sempre rilevante.
- **`search_base` e `search_v3`** abbassano l'MRR rispetto a semantic. *Il re-ranking
  spinge in alto prodotti meno rilevanti del top-1 puramente semantico.*
- **`search_v3`** è il peggiore — vedremo perché nelle ablation 3 e nell'analisi MRR.
- **`hybrid_v4`** ha il NDCG più alto in assoluto (0.526) ed è il candidato naturale
  come default production.


## 3. Ablation 1 — peso del re-ranking (β_quality, β_popularity)


In [ ]:
ablat_b = pd.read_csv(f"{ROOT}/data/eval_output/ablation_beta.csv")
print(ablat_b.round(4).to_string(index=False))


**Lettura**: i valori del team `(β_q=0.12, β_p=0.05)` sono vicini all'ottimo, ma
si scopre che si possono migliorare. β = (0.05, 0.02) → NDCG=0.511. Più aggressivo
(0.30, 0.15) deteriora NDCG di ~8 punti. **Il principio del re-ranking è giusto, i
pesi sono difendibili come "vicino all'ottimo testato"** — questo è quello che le
regole chiedono di documentare.


## 4. Ablation 2 — quality vs popularity isolati


In [ ]:
ablat_p = pd.read_csv(f"{ROOT}/data/eval_output/ablation_pop.csv")
print(ablat_p[["label","NDCG@5","MRR","Recall@5"]].round(4).to_string(index=False))


### Insight critico

Quando si tiene **solo il quality score** (β_p=0.00), TUTTE le metriche migliorano
rispetto al default attuale:
- NDCG@5: 0.5053 → **0.5232** (+1.8%)
- MRR: 0.7649 → **0.7832** (+1.8%)

**Il `popularity_score` sta facendo male.** Approfondiamo nella sezione 7.


## 5. Ablation 3 — moduli di `search_v3`


In [ ]:
v3_mod = pd.read_csv(f"{ROOT}/data/eval_output/v3_modules.csv")
print(v3_mod.round(4).to_string(index=False))


### Insight

Il problema di `search_v3` rispetto a `search_base` è il **`min_rating=3.5`** di
default: rimuovendolo (`min_rating=None`), `search_v3` torna alla pari di `search_base`
(NDCG 0.489 → 0.505).

Il filtro `min_rating` è una **scelta UX legittima** (l'utente non vede prodotti
brutti) ma penalizza la recall sui benchmark neutrali. Va presentato come trade-off
attivabile, non come default.


## 6. Per-intent — chi vince dove


In [ ]:
per_int_ndcg = pd.read_csv(f"{ROOT}/data/eval_output/per_intent_ndcg.csv", index_col=0)
per_int_ndcg["best"] = per_int_ndcg.drop(columns="best", errors="ignore").idxmax(axis=1)
print("=== NDCG@5 per intent ===")
print(per_int_ndcg.round(3).to_string())


In [ ]:
per_int_mrr = pd.read_csv(f"{ROOT}/data/eval_output/per_intent_mrr.csv", index_col=0)
print("=== MRR per intent ===")
print(per_int_mrr.round(3).to_string())


### Storyline da raccontare nelle slides

| Intent      | Vincitore (NDCG)  | Vincitore (MRR)  | Lezione |
|-------------|-------------------|------------------|---------|
| brand       | search_base       | (tie all)        | Match esatto basta — quality re-rank aiuta marginalmente |
| dosage      | **BM25**          | (tie all)        | Match lessicale è quasi tutto |
| price       | BM25, semantic    | BM25             | I filtri price hard del team penalizzano recall |
| negation    | search_base       | **semantic**     | Il negation filter di v3 non aggiunge valore vs base |
| semantic    | **hybrid_v4**     | semantic         | Linguaggio naturale → embedding |

> **Take-away presentazione**: *"BM25 vince sulle query lessicalmente esplicite
> (brand, dosaggio); embedding semantico vince sulle query in linguaggio naturale.
> Hybrid (BM25 + semantic) cattura entrambi i comportamenti — è il default che
> useremmo in produzione."*

Questo trasforma il "BM25 ci batte" da debolezza in **giustificazione** del perché
l'hybrid è la scelta architetturale corretta.


## 7. MRR top-1 analysis — chi sposta cosa?


In [ ]:
mrr_an = pd.read_csv(f"{ROOT}/data/eval_output/mrr_top1_analysis.csv")
print(mrr_an["verdict"].value_counts().to_string())
print()
regr = mrr_an[mrr_an["verdict"]=="BASE_REGRESSED"]
cols = ["qid","intent","sem_sim","base_sim","delta_quality","delta_popularity","sem_rel","base_rel"]
print("Casi dove search_base ha sostituito un top-1 rilevante con uno meno:")
print(regr[cols].round(3).to_string(index=False))


### Diagnosi

In **7 casi su 37** il re-ranking ha sostituito un top-1 rilevante con uno meno
rilevante (vs solo 1 caso opposto, ratio 7:1).

In **tutti i 7 casi**, il `delta_popularity` medio dello swap è **+0.5** (il prodotto
"vincitore" è molto più popolare), mentre il `delta_similarity` è negativo (il
"vincitore" è meno semanticamente vicino alla query).

**Conclusione**: il `popularity_score` con β=0.05 è il responsabile diretto della
regressione MRR di `search_base` rispetto a `semantic` puro. Il fix è azzerarlo.


## 8. Summarization & entity extraction — il valore *non-search*


In [ ]:
import json
with open(f"{ROOT}/data/eval_output/summarization_stats.json") as f:
    stats = json.load(f)
for k, v in stats.items():
    if isinstance(v, dict): print(f"{k}:"); [print(f"   {kk}: {vv}") for kk,vv in v.items()]
    else: print(f"{k}: {v}")


### Lettura

**Summarization è solida**:
- 92% dei prodotti hanno una sintesi BART; di questi, 80% sono generate dal modello,
  20% extractive (fallback per prodotti con poche review).
- Pros coverage 96%, Cons 71% — **asset distintivo**.
- Lunghezza media 142 chars (sintesi compatte e leggibili).

**Entity extraction è il pezzo debole**:
- Solo **8.8%** dei prodotti hanno almeno un ingrediente estratto.
- Il regex pattern hardcoded copre ~30 ingredienti — perde tutto il resto (zinc, iron,
  vitamine A/B/D/E/K, glucosamina, MSM, …).
- Certifications 20% — `natural` (1002 occorrenze) è troppo generico, possibile noise.

### Cosa raccontare nelle slides

> *"7,593 prodotti arricchiti con sintesi BART pro/cons (92%/96%/71%) + attributi
> strutturati (brand 53%, certificazioni 20%). Il sistema non è solo retrieval —
> è una pipeline di **review intelligence** che trasforma 318k review unstructured
> in profili prodotto utilizzabili."*

### Cosa migliorare prima della consegna del 10/5

Espandere il regex degli ingredienti in `summarization.py` per portare la coverage
da 9% a >30% richiede 30 minuti e migliora visibilmente i numeri della demo.


## 9. Guardrail — confusion matrix


In [ ]:
guard = pd.read_csv(f"{ROOT}/data/eval_output/guardrail_results.csv")
with open(f"{ROOT}/data/eval_output/guardrail_metrics.json") as f:
    m = json.load(f)
print("Confusion matrix (12 off-topic + 12 in-domain):")
print(f"               pred OFF | pred IN")
print(f"  exp OFF        TP={m['TP']:<3}   FN={m['FN']}")
print(f"  exp IN         FP={m['FP']:<3}   TN={m['TN']}")
print()
print(f"Accuracy:        {m['accuracy']:.3f}")
print(f"Precision block: {m['precision_block']:.3f}  (delle bloccate, quante davvero off-topic)")
print(f"Recall block:    {m['recall_block']:.3f}  (delle off-topic, quante effettivamente bloccate)")
print(f"F1 block:        {m['f1_block']:.3f}")
print()
print("Errori:")
print(guard[~guard["ok"]][["query","reason","confidence"]].to_string(index=False))


### Lettura

**F1 = 0.957** su 24 query. Il guardrail ha:
- **Precision = 1.000**: non ha *mai* bloccato una query in-domain per errore (zero falsi
  positivi). Importante per UX — niente utenti rifiutati ingiustamente.
- **Recall = 0.917**: 11 su 12 off-topic correttamente bloccate. L'unico errore è
  *"best italian recipes for pasta carbonara"* — il sistema l'ha lasciata passare con
  confidence 0.347 (basso). Fix banale: aggiungere `recipe|cooking|carbonara` alla
  blacklist regex.

> **Slide-ready**: *"Il guardrail intercetta il 92% delle query off-topic con
> zero falsi positivi. È il livello di sicurezza che differenzia un sistema
> sperimentale da uno production-ready."*


## 10. Patch concrete suggerite (file `SUGGESTED_PATCH.md`)

Tre modifiche con motivazione empirica:

| # | Patch | Effort | Effetto |
|---|-------|--------|---------|
| 1 | `BETA_POPULARITY = 0.0` in `config.py` | 1 char | NDCG +1.8%, MRR +1.8% (su tutti i sistemi) |
| 2 | `search_v3` default `min_rating=None` | 2 char | NDCG +1.6% MRR +2.2% (riallinea v3 con base) |
| 3 | README: rimuovere riferimento a `webapp/` inesistente | 1 min | Cosmetico (ma necessario) |
| 4 | `requirements.txt` con versioni pinnate | 5 min | Compliance regole submission |

**Vale la pena committare in commit separati e descrittivi prima della consegna del 10/5.**

## 11. Conclusioni — cosa raccontare a Deloitte

### 1. Il valore non è il retrieval da solo
`hybrid_v4` (BM25+semantic) è competitivo con tutti i baseline IR sull'evaluation
set. Ma il **valore distintivo** del progetto è la combinazione:
- retrieval review-aware (two-tower con dynamic alpha)
- 92% coverage di sintesi BART (pros/cons strutturati)
- entity extraction (brand/cert/ingredients) — *qui c'è ancora margine di miglioramento*
- guardrail con F1 0.957
- demo Gradio funzionante + UMAP visualization

### 2. Le scelte di design sono difendibili a posteriori
- `(β_q=0.12, β_p=0.05)` è vicino all'ottimo grid-searched (deve diventare `β_p=0.0` come da fix #1, ma il principio è giusto).
- Dynamic alpha `[0.35, 0.70]`: motivata teoricamente (review signal strength), non grid-searched ma ragionevole.
- Architettura two-tower: scelta originale, ben motivata, base di tutto il sistema.

### 3. Limiti dichiarati onestamente
- Pseudo-relevance benchmark (favorisce BM25)
- 37 query non sono sufficienti per conclusioni statisticamente robuste
- Dataset H&PC non contiene big brand (CeraVe, Neutrogena) — query brand-popolari falliscono
- Inglese only

### 4. Da fare entro il 10/5
- Applicare patch 1-4 (10 minuti, +0.034 NDCG)
- Espandere regex ingredients (+30 minuti, da 9% a 30% coverage)
- Annotare ulteriori 60-80 query con human relevance (3-4 ore divise nel team)
- Slide pack che racconta i tre lavori (retrieval, summarization, entity), non solo il retrieval
